# Comment Matching Analysis

This notebook analyzes comment-level alignment, guideline references, and qualitative examples in one top-to-bottom workflow.


In [ ]:
import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from scipy.optimize import linear_sum_assignment


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "code").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root.")


PROJECT_ROOT = find_project_root()
CODE_DIR = PROJECT_ROOT / "code"
RESULTS_DIR = CODE_DIR / "results" / "critiques" / "analysis"
MATCHING_DIR = CODE_DIR / "results" / "critiques" / "matching"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
QUALITATIVE_DIR = TABLES_DIR / "qualitative_examples"
for directory in [TABLES_DIR, FIGURES_DIR, QUALITATIVE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))
from utils.comments.guidelines import (
    APPLE_SUBCATEGORY_PARENT,
    APPLE_TOP_LOOKUP,
    CANONICAL_SOURCES,
    NIELSEN_CATEGORIES,
    NIELSEN_LOOKUP,
    build_parsed_long_df,
    clean_text,
    normalize_key,
    parse_guideline_references,
)
from utils.screen_display import show_screen

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "matplotlib"))

RUN_LABEL = "zero_shot-all_tasks"
MODEL_ORDER = ["claude4_7", "gpt5", "gemini3_1pro"]
MODEL_DISPLAY = {
    "claude4_7": "Claude Opus 4.7",
    "gpt5": "GPT-5",
    "gemini3_1pro": "Gemini 3.1 Pro",
}
MODEL_COLORS = {
    "claude4_7": "#F89838",
    "gpt5": "#5288C1",
    "gemini3_1pro": "#73C068",
}
MODEL_RANK = {model: rank for rank, model in enumerate(MODEL_ORDER)}

K_VALUES = [1, 3, 5]
BOOTSTRAP_ITERATIONS = 1000
RANDOM_SEED = 42

PAIR_KEYS = ["model_short", "screen_task_id", "expert_comment_id", "llm_comment_id"]
MODEL_GROUP_COLS = ["model_short", "model_name"]
EXPERT_GROUP_COLS = [*MODEL_GROUP_COLS, "screen_task_id", "expert_comment_id"]
SCREEN_GROUP_COLS = [*MODEL_GROUP_COLS, "app_name", "app_category", "screen_task_id"]

# Relation labels are canonicalized on load. Earlier matching runs (and the frozen
# qualitative review CSVs) emitted exact_match / llm_broader / human_broader, while later
# runs emit equivalent / B_broader / A_broader for the same three relations. Everything
# downstream - weights, ranking, sampling, manual coding - uses the canonical names only.
# RELATION_ORDER doubles as the ranking order, best match first.
RELATION_ORDER = [
    "equivalent",
    "llm_broader",
    "human_broader",
    "partial_overlap",
    "contradiction",
    "related_but_different",
    "no_match",
]
RELATION_RANK = {relation: rank for rank, relation in enumerate(RELATION_ORDER)}
RELATION_ALIASES = {
    "equivalent": "equivalent",
    "exact_match": "equivalent",
    "B_broader": "llm_broader",
    "llm_broader": "llm_broader",
    "A_broader": "human_broader",
    "human_broader": "human_broader",
    "partial_overlap": "partial_overlap",
    "contradiction": "contradiction",
    "related_but_different": "related_but_different",
    "no_match": "no_match",
    "no_relation": "no_match",
}
RELATION_WEIGHTS = {
    "equivalent": 1.00,
    "llm_broader": 0.85,
    "human_broader": 0.85,
    "partial_overlap": 0.50,
    "contradiction": 0.00,
    "related_but_different": 0.00,
    "no_match": 0.00,
}


def canonical_relation(values: pd.Series) -> pd.Series:
    """Map raw relation labels from any matching run onto RELATION_ORDER."""
    unknown = sorted(set(values.dropna().unique()) - set(RELATION_ALIASES))
    if unknown:
        raise ValueError(f"Unrecognized relation labels: {unknown}")
    return values.map(RELATION_ALIASES)


def display_df(df: pd.DataFrame) -> None:
    display(df.drop(columns=["model_short"], errors="ignore"))


def model_color(display_name: str) -> str:
    return MODEL_COLORS[next(model for model in MODEL_ORDER if MODEL_DISPLAY[model] == display_name)]


def save_figure(filename: str, fig=None) -> None:
    (fig or plt.gcf()).savefig(FIGURES_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


sns.set_theme(context="paper", style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 1. Load and Prepare Matching Data


In [ ]:
ARTIFACTS = {
    "all_pairs": "all_comment_pairs-{stem}.parquet",
    "pair_labels": "llmjudge_matching/llmjudge_matching_pairs-{stem}.parquet",
}

# GPT-5 returned an empty critique list for these two tasks, so they have no candidate
# pairs and are dropped from every model to keep the comparison on a shared task set.
EXCLUDED_SCREEN_TASK_IDS = {"20234_T03", "31563_T02"}


def load_artifact(model: str, artifact: str) -> pd.DataFrame:
    stem = f"{RUN_LABEL}-{model}"
    path = MATCHING_DIR / model / RUN_LABEL / ARTIFACTS[artifact].format(stem=stem)
    df = pd.read_parquet(path)
    df["model_short"] = model
    df["model_name"] = MODEL_DISPLAY[model]
    return df


def load_all(artifact: str) -> pd.DataFrame:
    df = pd.concat([load_artifact(model, artifact) for model in MODEL_ORDER], ignore_index=True)
    return df[~df["screen_task_id"].isin(EXCLUDED_SCREEN_TASK_IDS)].reset_index(drop=True)


all_pairs_df = load_all("all_pairs")
pair_labels_df = load_all("pair_labels")
pair_labels_df["relation"] = canonical_relation(pair_labels_df["relation"])

human_tasks = (
    pd.read_parquet(PROJECT_ROOT / "data" / "dataset" / "cleaned_dataset" / "human_comments_semantic_dedup.parquet")
    [["screen_task_id", "app", "app_category"]]
    .rename(columns={"app": "app_name"})
    .drop_duplicates("screen_task_id")
)

# `pairs` is the weighted pair table used by the cardinality-controlled metrics.
pairs = pair_labels_df.merge(human_tasks, on="screen_task_id", how="left")
pairs["relation_weight"] = pairs["relation"].map(RELATION_WEIGHTS).astype(float)

# Preserve the original generated-comment order through first appearance in the pair table.
llm_order = (
    pairs[["model_short", "screen_task_id", "llm_comment_id"]]
    .drop_duplicates()
    .assign(llm_position=lambda x: x.groupby(["model_short", "screen_task_id"]).cumcount() + 1)
)
pairs = pairs.merge(llm_order, on=["model_short", "screen_task_id", "llm_comment_id"], how="left")

load_summary = (
    pairs.groupby(MODEL_GROUP_COLS, sort=False)
    .agg(
        pair_rows=("pair_id", "size"),
        screen_tasks=("screen_task_id", "nunique"),
        expert_comments=("expert_comment_id", "nunique"),
        llm_comments=("llm_comment_id", "nunique"),
        apps=("app_name", "nunique"),
    )
    .reset_index()
)
load_summary

### 1.1 Prepare Pair-Level Metrics and Expert-Coverage Selections

Enrich selected relation rows with automatic metrics and derive one best match per expert comment for the qualitative analyses.


In [ ]:
# `metric_pairs_df` is the pair table with expert/LLM text and screen context attached;
# the qualitative sections read their examples from it.
pair_context_columns = [col for col in all_pairs_df.columns if col not in {"pair_id", "model_name"}]
metric_pairs_df = (
    pair_labels_df[[*PAIR_KEYS, "model_name", "pair_id", "relation", "confidence"]].drop_duplicates(PAIR_KEYS)
    .merge(
        all_pairs_df[pair_context_columns].drop_duplicates(PAIR_KEYS),
        on=PAIR_KEYS,
        how="left",
        validate="one_to_one",
    )
)


def best_ranked_relation(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Keep the single best-ranked (then highest-confidence) row per group."""
    ranked = df.assign(relation_rank=df["relation"].map(RELATION_RANK))
    ranked = ranked[ranked["relation_rank"].notna()]
    return (
        ranked.sort_values(
            [*group_cols, "relation_rank", "confidence"],
            ascending=[True] * len(group_cols) + [True, False],
        )
        .drop_duplicates(group_cols)
        .drop(columns="relation_rank")
    )


# One selected match per expert comment; drives the qualitative coverage analyses.
expert_coverage_df = (
    all_pairs_df[EXPERT_GROUP_COLS].drop_duplicates()
    .merge(
        best_ranked_relation(pair_labels_df, EXPERT_GROUP_COLS)[
            [*EXPERT_GROUP_COLS, "llm_comment_id", "relation", "confidence"]
        ].rename(columns={"relation": "best_relation", "confidence": "best_confidence"}),
        on=EXPERT_GROUP_COLS,
        how="left",
    )
)
display_df(expert_coverage_df.head(3))

### 1.2 Input Validation

A single pass over the loaded tables. Structural problems (missing columns, unaligned
models, duplicate keys) raise; unexpected nulls are reported as warnings so that they
surface early instead of showing up as silent gaps in a downstream table.


In [ ]:
def validate_inputs() -> None:
    """Raise on structural problems; warn on anything merely surprising."""
    required = {"pair_id", *PAIR_KEYS, "relation", "confidence", "expert_observed_issue", "llm_observed_issue"}
    task_sets = {m: set(all_pairs_df.loc[all_pairs_df["model_short"].eq(m), "screen_task_id"]) for m in MODEL_ORDER}

    errors = []
    if missing := sorted(required - set(pair_labels_df.columns)):
        errors.append(f"pair labels missing columns: {missing}")
    if leaked := sorted(EXCLUDED_SCREEN_TASK_IDS & set(all_pairs_df["screen_task_id"])):
        errors.append(f"excluded tasks still present: {leaked}")
    if differing := [m for m, t in task_sets.items() if t != task_sets[MODEL_ORDER[0]]]:
        errors.append(f"task set differs for: {differing}")
    if misaligned := [m for m in MODEL_ORDER
                      if all_pairs_df["model_short"].eq(m).sum() != pair_labels_df["model_short"].eq(m).sum()]:
        errors.append(f"pair/label row counts differ for: {misaligned}")
    if pair_labels_df.duplicated(["model_short", "pair_id"]).any():
        errors.append("duplicate pair_id within a model")
    if unmapped := int(metric_pairs_df["relation"].isna().sum()):
        errors.append(f"{unmapped:,} pairs without a relation")
    if orphaned := int(metric_pairs_df["expert_observed_issue"].isna().sum()):
        errors.append(f"{orphaned:,} pairs without expert text")
    if "llm_guideline_reference" not in all_pairs_df.columns:
        errors.append("all_pairs is missing llm_guideline_reference")
    if errors:
        raise ValueError("Input validation failed:\n- " + "\n- ".join(errors))

    nulls = {f"{name}.{col}": int(df[col].isna().sum())
             for name, df, cols in [("labels", pair_labels_df, ["confidence", "expert_observed_issue", "llm_observed_issue"]),
                                    ("pairs", all_pairs_df, ["screen_id", "task", "app_category"]),
                                    ("pairs", pairs, ["app_name"])]
             for col in cols if df[col].isna().any()}
    if nulls:
        print("warning - unexpected nulls:", "; ".join(f"{k}={v:,}" for k, v in nulls.items()))

    print(f"Validation passed: {len(task_sets[MODEL_ORDER[0]]):,} shared screen tasks across {len(MODEL_ORDER)} models.")


validate_inputs()

## 2. Macro-Averaged Unconstrained and Bipartite Metrics


In [ ]:
def _assignment_scores(weights: np.ndarray, expert_count: int, llm_count: int) -> tuple[float, float, float]:
    if expert_count == 0 or llm_count == 0:
        return np.nan, np.nan, 0.0
    row_ind, col_ind = linear_sum_assignment(-weights)
    matched_sum = float(weights[row_ind, col_ind].sum())
    return matched_sum / expert_count, matched_sum / llm_count, matched_sum


def _weighted_matrix(screen_pairs: pd.DataFrame) -> pd.DataFrame:
    llm_ordered = (
        screen_pairs[["llm_comment_id", "llm_position"]]
        .drop_duplicates()
        .sort_values("llm_position")["llm_comment_id"]
        .tolist()
    )
    return screen_pairs.pivot_table(
        index="expert_comment_id",
        columns="llm_comment_id",
        values="relation_weight",
        aggfunc="max",
        fill_value=0.0,
    ).reindex(columns=llm_ordered, fill_value=0.0)


screen_rows = []
k_screen_rows = []

for keys, screen_pairs in pairs.groupby(SCREEN_GROUP_COLS, sort=False):
    key_values = dict(zip(SCREEN_GROUP_COLS, keys))
    matrix = _weighted_matrix(screen_pairs)
    weights = matrix.to_numpy(dtype=float)
    expert_count, llm_count = matrix.shape

    bip_cov, bip_sup, matched_sum = _assignment_scores(weights, expert_count, llm_count)
    screen_rows.append({
        **key_values,
        "unconstrained_expert_coverage": float(weights.max(axis=1).mean()) if expert_count else np.nan,
        "unconstrained_reference_support": float(weights.max(axis=0).mean()) if llm_count else np.nan,
        "bipartite_expert_coverage": bip_cov,
        "bipartite_reference_support": bip_sup,
        "matched_weight_sum": matched_sum,
        "expert_comments": expert_count,
        "llm_comments": llm_count,
    })

    positions = (
        screen_pairs[["llm_comment_id", "llm_position"]]
        .drop_duplicates()
        .set_index("llm_comment_id")
        .loc[matrix.columns, "llm_position"]
        .to_numpy()
    )
    for k in K_VALUES:
        keep = positions <= k
        k_cov, k_sup, _ = _assignment_scores(weights[:, keep], expert_count, int(keep.sum()))
        k_screen_rows.append({**key_values, "k": k, "bipartite_expert_coverage": k_cov, "bipartite_reference_support": k_sup})

screen_metrics = pd.DataFrame(screen_rows)
k_screen_metrics = pd.DataFrame(k_screen_rows)

core_metric_cols = [
    "unconstrained_expert_coverage",
    "unconstrained_reference_support",
    "bipartite_expert_coverage",
    "bipartite_reference_support",
]


def as_percent(df: pd.DataFrame, metric_cols: list[str]) -> pd.DataFrame:
    """Rename metric columns to *_pct and scale them to percentages."""
    out = df.rename(columns={metric: f"{metric}_pct" for metric in metric_cols}).copy()
    out[[f"{metric}_pct" for metric in metric_cols]] *= 100
    return out.round(1)


core_metrics_pct = as_percent(
    screen_metrics.groupby(MODEL_GROUP_COLS, sort=False)[core_metric_cols].mean().reset_index(),
    core_metric_cols,
)
screen_metrics.round(2).to_parquet(TABLES_DIR / "screen_level_alignment_metrics.parquet", index=False)
core_metrics_pct

## 3. Clustered App-Level Bootstrap Confidence Intervals


In [ ]:
def bootstrap_macro_ci(
    screen_df: pd.DataFrame,
    metric_cols: list[str],
    *,
    iterations: int = BOOTSTRAP_ITERATIONS,
    seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    """Resample apps (not screens) within each model to respect app-level clustering."""
    rng = np.random.default_rng(seed)
    rows = []

    for keys, model_df in screen_df.groupby(MODEL_GROUP_COLS, sort=False):
        key_dict = dict(zip(MODEL_GROUP_COLS, keys))
        app_names = model_df["app_name"].dropna().unique()
        by_app = {app: app_df[metric_cols].to_numpy(dtype=float) for app, app_df in model_df.groupby("app_name", sort=False)}

        boot_values = np.empty((iterations, len(metric_cols)), dtype=float)
        for i in range(iterations):
            sampled_apps = rng.choice(app_names, size=len(app_names), replace=True)
            boot_values[i] = np.nanmean(np.vstack([by_app[app] for app in sampled_apps]), axis=0)

        for j, metric in enumerate(metric_cols):
            low, high = np.percentile(boot_values[:, j], [2.5, 97.5])
            rows.append({
                **key_dict,
                "metric": metric,
                "point_estimate": float(model_df[metric].mean()),
                "ci_low": float(low),
                "ci_high": float(high),
                "bootstrap_iterations": iterations,
            })

    return pd.DataFrame(rows)


core_ci = bootstrap_macro_ci(screen_metrics, core_metric_cols)

core_ci_pct = core_ci.rename(columns={"point_estimate": "point_estimate_pct", "ci_low": "ci_low_pct", "ci_high": "ci_high_pct"})
core_ci_pct[["point_estimate_pct", "ci_low_pct", "ci_high_pct"]] *= 100
core_ci_pct = core_ci_pct.round(1)
core_ci_pct.to_csv(TABLES_DIR / "macro_core_alignment_bootstrap_ci.csv", index=False, float_format="%.1f")
display(core_ci_pct)

core_metric_groups = {
    "Expert Coverage": {"unconstrained": "unconstrained_expert_coverage", "bipartite": "bipartite_expert_coverage"},
    "Reference Support": {"unconstrained": "unconstrained_reference_support", "bipartite": "bipartite_reference_support"},
}
method_labels = {"unconstrained": "Unconstrained", "bipartite": "Bipartite"}
method_alpha = {"unconstrained": 0.45, "bipartite": 0.95}

bar_width = 0.24
x_centers = np.arange(len(MODEL_ORDER), dtype=float)
method_offsets = {"unconstrained": -(bar_width + 0.03) / 2, "bipartite": (bar_width + 0.03) / 2}

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.8), sharey=True)
for ax, (panel_title, metrics) in zip(axes, core_metric_groups.items()):
    for model_idx, model in enumerate(MODEL_ORDER):
        color = MODEL_COLORS[model]
        model_rows = core_ci[core_ci["model_short"].eq(model)].set_index("metric")

        for method, metric in metrics.items():
            row = model_rows.loc[metric]
            x = x_centers[model_idx] + method_offsets[method]
            y = row["point_estimate"]
            ax.bar(x, y, width=bar_width, color=color, alpha=method_alpha[method], edgecolor=color, linewidth=0.8)
            ax.errorbar(
                x, y,
                yerr=[[y - row["ci_low"]], [row["ci_high"] - y]],
                fmt="none", color="black", capsize=3, linewidth=1,
            )

        ax.plot(
            [x_centers[model_idx] + method_offsets[method] for method in method_labels],
            [model_rows.loc[metrics[method], "point_estimate"] for method in method_labels],
            marker="o", markersize=3.5, color=color, alpha=0.7, linewidth=1,
        )

    ax.set_title(panel_title, fontsize=12, weight="bold")
    ax.set_xticks(x_centers)
    ax.set_xticklabels([MODEL_DISPLAY[model] for model in MODEL_ORDER], rotation=45, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))

axes[0].set_ylabel("Macro score (%)")
axes[0].set_ylim(0, 1)
fig.legend(
    handles=[plt.Rectangle((0, 0), 1, 1, color="gray", alpha=method_alpha[m], label=method_labels[m]) for m in method_labels],
    loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=2, frameon=False,
)
fig.tight_layout(rect=(0, 0, 1, 0.95))
save_figure("macro_core_alignment_metrics_barplot.png", fig)

## 4. Fixed-K Bipartite Budget Control

At each threshold $K$, bipartite expert coverage and reference support are recomputed using only the first $K$ generated critiques for each screen--task instance.

In [ ]:
fixed_k_metric_cols = ["bipartite_expert_coverage", "bipartite_reference_support"]
fixed_k_metrics = (
    k_screen_metrics.groupby([*MODEL_GROUP_COLS, "k"], sort=False)[fixed_k_metric_cols].mean().reset_index()
)

fixed_k_metrics_pct = as_percent(fixed_k_metrics, fixed_k_metric_cols)
fixed_k_metrics_pct.to_csv(TABLES_DIR / "fixed_k_macro_bipartite_metrics.csv", index=False, float_format="%.1f")
display(fixed_k_metrics_pct)

fig, ax = plt.subplots(figsize=(7.2, 5.2))
for model in MODEL_ORDER:
    model_line = fixed_k_metrics[fixed_k_metrics["model_short"].eq(model)].sort_values("k")
    ax.plot(
        model_line["bipartite_reference_support"],
        model_line["bipartite_expert_coverage"],
        marker="o", linewidth=2, color=MODEL_COLORS[model], label=MODEL_DISPLAY[model],
    )
    for row in model_line.itertuples(index=False):
        ax.annotate(
            f"K={row.k}",
            (row.bipartite_reference_support, row.bipartite_expert_coverage),
            xytext=(4, 4), textcoords="offset points", fontsize=8, color=MODEL_COLORS[model],
        )
ax.set_xlabel("Reference Support (%)")
ax.set_ylabel("Expert Coverage (%)")
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.set_xlim(0, 0.4)
ax.set_ylim(0, 0.4)
ax.grid(linestyle="--", alpha=0.3)
ax.legend(title="Model", frameon=False)
fig.tight_layout()
save_figure("fixed_k_bipartite_coverage_vs_support.png", fig)

## 5. Contradiction Analysis


In [ ]:
contradiction_summary = (
    pairs.assign(is_contradiction=pairs["relation"].eq("contradiction"))
    .groupby(MODEL_GROUP_COLS, sort=False)
    .apply(
        lambda g: pd.Series({
            "llm_comments": g["llm_comment_id"].nunique(),
            "contradiction_pairs": int(g["is_contradiction"].sum()),
            "llm_comments_with_contradiction": g.loc[g["is_contradiction"], "llm_comment_id"].nunique(),
        }),
        include_groups=False,
    )
    .reset_index()
)
contradiction_summary["contradiction_llm_comment_rate_pct"] = 100 * (
    contradiction_summary["llm_comments_with_contradiction"] / contradiction_summary["llm_comments"]
)
contradiction_summary = contradiction_summary.round(2)
contradiction_summary.to_csv(TABLES_DIR / "contradiction_summary.csv", index=False, float_format="%.2f")
contradiction_summary


## 6. Guideline Reference Analysis

This section analyzes whether LLM critiques include guideline references and whether referenced critiques are more grounded in expert critiques.

Run this section only if `llm_guideline_reference` exists in the LLM-comment table. The unit of analysis is the unique LLM comment, not the pair row, so each generated critique contributes once.


### 6.1 Parse References

Reference strings are parsed by `utils.comments.guidelines`, which normalizes source names,
Nielsen heuristics and Apple HIG concepts into one record per cited guideline.

The check below lists any wording the parser could not resolve; new entries belong in
`utils.comments.guidelines`.

In [ ]:
refs_df = (
    all_pairs_df[["model_short", "model_name", "llm_comment_id", "llm_guideline_reference"]]
    .drop_duplicates(["model_short", "llm_comment_id"])
    .copy()
)
refs_df["parsed_guideline_references"] = refs_df["llm_guideline_reference"].apply(parse_guideline_references)
parsed_long_df = build_parsed_long_df(refs_df)

print(f"LLM comments: {len(refs_df):,}")
print(f"Missing reference field: {int(refs_df['llm_guideline_reference'].isna().sum()):,}")
print(f"Parsed references: {len(parsed_long_df):,}")

In [ ]:
def unmapped_reference_wordings(top_n: int = 15) -> pd.DataFrame:
    """Guideline wordings the parser could not resolve, most frequent first.

    An empty result means every reference mapped onto a known source, heuristic or
    Apple HIG concept. Anything listed needs a new alias in utils.comments.guidelines.
    """
    is_nielsen = parsed_long_df["source"].eq("Nielsen")
    is_apple = parsed_long_df["source"].eq("Apple HIG")
    checks = [
        ("unknown source", parsed_long_df["source"], CANONICAL_SOURCES),
        ("unmapped Nielsen category", parsed_long_df["category"].where(is_nielsen), NIELSEN_LOOKUP),
        ("unmapped Apple category", parsed_long_df["category"].where(is_apple), APPLE_TOP_LOOKUP),
        ("unmapped Apple subcategory", parsed_long_df["subcategory"].where(is_apple), APPLE_SUBCATEGORY_PARENT),
    ]

    found = [
        pd.DataFrame({
            "issue": "unparsed reference",
            "value": refs_df.loc[
                refs_df["llm_guideline_reference"].notna() & refs_df["parsed_guideline_references"].str.len().eq(0),
                "llm_guideline_reference",
            ],
        })
    ]
    for issue, values, known in checks:
        values = values.dropna()
        unmapped = values[~values.map(lambda v: (v if issue == "unknown source" else normalize_key(v)) in known)]
        found.append(pd.DataFrame({"issue": issue, "value": unmapped}))

    summary = (
        pd.concat(found)
        .groupby(["issue", "value"], as_index=False)
        .size()
        .sort_values(["issue", "size"], ascending=[True, False])
        .rename(columns={"size": "references"})
    )
    if summary.empty:
        print("No unmapped guideline wordings.")
    return summary.groupby("issue", group_keys=False).head(top_n).reset_index(drop=True)


display(unmapped_reference_wordings())

### 6.2 Reference Presence

Check that every unique LLM comment has a guideline-reference field and at least one parsed reference.


In [ ]:
reference_presence_df = refs_df.assign(
    has_guideline_reference=refs_df["llm_guideline_reference"].notna()
    & refs_df["llm_guideline_reference"].astype(str).str.strip().ne(""),
    parsed_reference_count=refs_df["parsed_guideline_references"].str.len(),
)
reference_presence_df["parsed_reference_count_when_present"] = (
    reference_presence_df["parsed_reference_count"].where(reference_presence_df["has_guideline_reference"])
)
reference_presence_df["unparsed_nonmissing_reference"] = (
    reference_presence_df["has_guideline_reference"] & reference_presence_df["parsed_reference_count"].eq(0)
)

guideline_reference_presence_df = (
    reference_presence_df.groupby(MODEL_GROUP_COLS, observed=True)
    .agg(
        llm_comments=("llm_comment_id", "nunique"),
        comments_with_reference=("has_guideline_reference", "sum"),
        min_parsed_references_when_present=("parsed_reference_count_when_present", "min"),
        unparsed_nonmissing_references=("unparsed_nonmissing_reference", "sum"),
    )
    .reset_index()
)
guideline_reference_presence_df["missing_references"] = (
    guideline_reference_presence_df["llm_comments"] - guideline_reference_presence_df["comments_with_reference"]
)
guideline_reference_presence_df = (
    guideline_reference_presence_df
    .sort_values("model_short", key=lambda s: s.map(MODEL_RANK))
    .reset_index(drop=True)
)

assert guideline_reference_presence_df["unparsed_nonmissing_references"].eq(0).all()
display(guideline_reference_presence_df)


### 6.3 Guideline Focus

This section stays at the unique LLM-comment/reference level and summarizes which guideline concepts were repeatedly cited. All three figures use grouped horizontal bars, consistent model colors, and direct percentage labels so that models can be compared on a common baseline. The category construction remains framework-specific: Nielsen uses its fixed canonical taxonomy, Apple HIG uses a compact set of shared concept bins, and CrowdCrit uses broader focus groups derived from its noisier free-text labels.


In [ ]:
def reference_percentage_matrix(df: pd.DataFrame, label_col: str, row_order: list[str]) -> pd.DataFrame:
    """Percentage of each model's references falling in each label, as a label x model matrix."""
    counts = (
        df.dropna(subset=[label_col])
        .groupby([*MODEL_GROUP_COLS, label_col], observed=True)
        .size()
        .rename("reference_count")
        .reset_index()
    )
    counts["pct_references"] = 100 * counts["reference_count"] / counts.groupby(
        "model_short", observed=True
    )["reference_count"].transform("sum")
    return (
        counts.pivot(index=label_col, columns="model_name", values="pct_references")
        .reindex(index=row_order, columns=[MODEL_DISPLAY[model] for model in MODEL_ORDER])
        .fillna(0)
    )


def plot_reference_percentages(plot_df, title, xlabel, filename, figsize=(11, 7.2), legend_loc="best"):
    ax = plot_df.plot.barh(figsize=figsize, width=0.78, color=[model_color(column) for column in plot_df.columns])
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    ax.grid(axis="x", linestyle="--", alpha=0.3)
    ax.set_axisbelow(True)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f", padding=2, fontsize=8)
    ax.legend(title="Model", loc=legend_loc, frameon=False)
    plt.tight_layout()
    save_figure(filename)


#### 6.3.1 Nielsen Heuristic Distribution

The figure presents Nielsen's canonical 10 heuristics in their original numbered order. Each row contains one bar per model, and percentages are calculated within each model's successfully mapped Nielsen references. Non-canonical Nielsen-attributed labels are excluded from this denominator and retained in the parser diagnostics. The shared horizontal scale supports direct comparison while preserving the complete heuristic names.


In [ ]:
nielsen_refs_df = parsed_long_df.loc[
    parsed_long_df["source"].eq("Nielsen") & parsed_long_df["category"].isin(NIELSEN_CATEGORIES)
]
nielsen_plot_df = reference_percentage_matrix(nielsen_refs_df, "category", NIELSEN_CATEGORIES)

display(nielsen_plot_df.round(2))
plot_reference_percentages(
    nielsen_plot_df,
    "Nielsen Heuristic References by Model",
    "% of mapped Nielsen references",
    "guideline_nielsen.png",
    legend_loc="lower right",
)


#### 6.3.2 Apple HIG Concept Distribution

For reporting, Apple HIG category and subcategory are merged into one derived concept label when the subcategory is known; the underlying parser output is unchanged. The figure uses the nine mapped concepts with the largest pooled reference counts across all three models, ordered by that pooled frequency. A tenth `Other (including unmapped / non-canonical)` bin combines every remaining mapped concept with all unmapped labels. Consequently, the 10 bins are exhaustive and each model's percentages sum to 100% before rounding. The shared bins and horizontal scale permit direct comparisons across models.


In [ ]:
APPLE_TOP_K = 9
APPLE_OTHER_LABEL = "Other (including unmapped / non-canonical)"


def concept_text(value):
    return None if pd.isna(value) else (clean_text(value) or None)


apple_df = parsed_long_df.loc[parsed_long_df["source"].eq("Apple HIG")].copy()
apple_df["apple_concept"] = [
    " / ".join(part for part in (concept_text(category), concept_text(subcategory)) if part) or None
    for category, subcategory in zip(apple_df["category"], apple_df["subcategory"])
]

apple_is_mapped = apple_df["category"].apply(lambda value: normalize_key(value) in APPLE_TOP_LOOKUP)
apple_top_concepts = (
    apple_df.loc[apple_is_mapped, "apple_concept"].value_counts().head(APPLE_TOP_K).index.tolist()
)
apple_df["apple_plot_concept"] = apple_df["apple_concept"].where(
    apple_is_mapped & apple_df["apple_concept"].isin(apple_top_concepts),
    APPLE_OTHER_LABEL,
)

apple_plot_df = reference_percentage_matrix(
    apple_df, "apple_plot_concept", [*apple_top_concepts, APPLE_OTHER_LABEL]
)
display(apple_plot_df.round(2))
plot_reference_percentages(
    apple_plot_df,
    "Apple HIG References by Concept and Model",
    "% of Apple HIG references",
    "guideline_apple_hig.png",
    figsize=(12, 7.2),
)


#### 6.3.3 CrowdCrit Category Distribution

CrowdCrit's noisier free-text labels are merged into broader focus groups using the documented keyword rules above. The figure includes every grouped reference, orders the substantive focus groups by their pooled frequency across all three models, and places `Other / specific critique labels` last. Each row contains one bar per model; percentages are calculated within each model's CrowdCrit references and sum to 100% before rounding.


In [ ]:
CROWDCRIT_OTHER_LABEL = "Other / specific critique labels"
CROWDCRIT_FOCUS_PATTERNS = {
    "Color, Contrast & Legibility": r"contrast|color|legib",
    "Visual Hierarchy & Emphasis": r"hierarch|emphasis|weight",
    "Typography & Text": r"typograph|text|readab",
    "Layout, Spacing & Alignment": r"layout|space|align|proportion|size",
    "Labeling, Content & Information Clarity": r"label|content|inform|clarity",
    "Gestalt: Grouping & Proximity": r"group|proximit|gestalt|unity|organi|differentiat",
    "Affordance & Usability": r"afford|usab|button|ui element|principle",
    "Iconography & Imagery": r"icon|imag",
    "Consistency & Standards": r"consisten|style|aesthetic|visual design",
    "Clutter & Simplicity": r"clutter|noise|simplic|redund",
}


def crowdcrit_focus(value):
    key = normalize_key(value)
    return next(
        (focus for focus, pattern in CROWDCRIT_FOCUS_PATTERNS.items() if re.search(pattern, key)),
        CROWDCRIT_OTHER_LABEL,
    )


crowdcrit_df = parsed_long_df.loc[parsed_long_df["source"].eq("CrowdCrit")].copy()
crowdcrit_df["crowdcrit_focus"] = crowdcrit_df["category"].apply(crowdcrit_focus)

crowdcrit_focus_order = (
    crowdcrit_df.loc[crowdcrit_df["crowdcrit_focus"].ne(CROWDCRIT_OTHER_LABEL), "crowdcrit_focus"]
    .value_counts()
    .index
    .tolist()
)
crowdcrit_plot_df = reference_percentage_matrix(
    crowdcrit_df, "crowdcrit_focus", [*crowdcrit_focus_order, CROWDCRIT_OTHER_LABEL]
)
display(crowdcrit_plot_df.round(2))
plot_reference_percentages(
    crowdcrit_plot_df,
    "CrowdCrit Reference Focus by Model",
    "% of CrowdCrit references",
    "guideline_crowdcrit.png",
    figsize=(12, 7.2),
)


## 7. Qualitative Analysis

Notes on the generated critiques:
- LLMs write detailed comments
- most of it is about visuals
- LLMs focus on atomic things, whereas humans talk about multiple things or generalize rather than being specific

### 7.1 Pair Review Setup

Shared machinery for the three pair-level manual reviews (7.2, 7.4, 7.5). Each review draws a
frozen sample of 30 pairs per model, stratified by `app_category`, and records one verdict per
pair in `pair_review_labels.csv` keyed by `(section, model_short, sample_index)`.

In [ ]:
REVIEW_SAMPLE_SIZE = 30
PAIR_REVIEW_TARGET = REVIEW_SAMPLE_SIZE * len(MODEL_ORDER)
PAIR_REVIEW_LABELS_PATH = QUALITATIVE_DIR / "pair_review_labels.csv"

# One frozen sample file per reviewed relation. Stable section keys: earlier runs stored the
# notebook section number instead, which changed whenever the notebook was renumbered.
REVIEW_SECTIONS = {
    "matched": {"matched", "equivalent"},
    "unsupported": {"unsupported", "11.3", "7.4", "7.3"},
    "contradiction": {"contradiction", "11.4", "7.5", "7.4"},
}
REVIEW_RELATIONS = {
    "matched": "equivalent",
    "unsupported": "no_match",
    "contradiction": "contradiction",
}
REVIEW_SAMPLE_PATHS = {
    "matched": QUALITATIVE_DIR / "matched_review_sample.csv",
    "unsupported": QUALITATIVE_DIR / "unsupported_comment_review_sample_old_pairs.csv",
    "contradiction": QUALITATIVE_DIR / "contradiction_review_sample.csv",
}
SECTION_ALIASES = {alias: name for name, aliases in REVIEW_SECTIONS.items() for alias in aliases}


def stratified_sample_pairs(df, n=15, seed=RANDOM_SEED):
    """Take one pair per app category first, then top up at random to n rows."""
    # Dedupe on pair_id: some context columns (e.g. expert_bounding_box) are unhashable.
    df = df.dropna(subset=["llm_observed_issue"])
    df = df.drop_duplicates(subset="pair_id" if "pair_id" in df.columns else None).copy()
    if df.empty:
        return df

    df["app_category"] = df["app_category"].fillna("Unknown")
    n = min(n, len(df))

    sample_df = pd.concat(
        [group_df.sample(1, random_state=seed) for _, group_df in df.groupby("app_category", sort=True)]
    )
    sample_df = sample_df.sample(frac=1, random_state=seed).head(n)

    remaining_n = n - len(sample_df)
    if remaining_n > 0:
        remaining_df = df.drop(index=sample_df.index)
        sample_df = pd.concat([sample_df, remaining_df.sample(n=min(remaining_n, len(remaining_df)), random_state=seed)])

    return sample_df.sample(frac=1, random_state=seed).reset_index(drop=True)


def load_or_create_pair_review_sample(section, n=REVIEW_SAMPLE_SIZE, seed=RANDOM_SEED):
    """Return the frozen review sample for a section, creating it once if absent."""
    sample_path = REVIEW_SAMPLE_PATHS[section]
    if sample_path.exists():
        sample_df = pd.read_csv(sample_path)
        sample_df["relation"] = canonical_relation(sample_df["relation"])
        return sample_df

    # Never silently re-draw a sample that recorded verdicts point into: they are keyed by
    # sample_index, so a new draw would silently reattach them to different pairs.
    if PAIR_REVIEW_LABELS_PATH.exists():
        saved_df = pd.read_csv(PAIR_REVIEW_LABELS_PATH, dtype={"section": str})
        sections = saved_df["section"].map(SECTION_ALIASES).fillna(saved_df["section"])
        recorded = int(sections.eq(section).sum())
        if recorded:
            raise FileNotFoundError(
                f"{sample_path} is missing, but {recorded} {section} verdicts reference it by "
                "sample_index. Restore the file instead of re-drawing the sample."
            )

    cols = [
        "model_short", "model_name", "pair_id", "screen_task_id",
        "expert_comment_id", "llm_comment_id", "screen_id", "app_category",
        "task", "relation", "confidence", "expert_observed_issue",
        "expert_bounding_box", "llm_observed_issue",
    ]
    source_df = metric_pairs_df.loc[metric_pairs_df["relation"].eq(REVIEW_RELATIONS[section]), cols]
    samples = []
    for model in MODEL_ORDER:
        model_sample_df = stratified_sample_pairs(source_df[source_df["model_short"].eq(model)], n=n, seed=seed)
        model_sample_df.insert(2, "sample_index", range(len(model_sample_df)))
        samples.append(model_sample_df)

    sample_df = pd.concat(samples, ignore_index=True)
    sample_df.to_csv(sample_path, index=False)
    print(f"Froze {len(sample_df):,} sampled pairs to {sample_path}")
    return sample_df


def add_expert_bounding_boxes(sample_df):
    """Backfill expert bounding boxes for frozen samples that predate the column."""
    if "expert_bounding_box" in sample_df and sample_df["expert_bounding_box"].notna().all():
        return sample_df.copy()

    result_df = sample_df.drop(columns="expert_bounding_box", errors="ignore").copy()
    result_df["_expert_id_norm"] = result_df["expert_comment_id"].astype("string").str.removeprefix("H_")

    bbox_lookup_df = metric_pairs_df[
        ["model_short", "screen_task_id", "expert_comment_id", "expert_bounding_box"]
    ].copy()
    bbox_lookup_df["_expert_id_norm"] = bbox_lookup_df["expert_comment_id"].astype("string").str.removeprefix("H_")
    bbox_lookup_df = bbox_lookup_df.drop(columns="expert_comment_id").drop_duplicates(
        ["model_short", "screen_task_id", "_expert_id_norm"]
    )

    return result_df.merge(
        bbox_lookup_df,
        on=["model_short", "screen_task_id", "_expert_id_norm"],
        how="left",
        validate="many_to_one",
    ).drop(columns="_expert_id_norm")


def show_comment_for_review(section, model="claude4_7", index=0):
    """Display one sampled pair and remember it for the next save_current_pair_review call."""
    global current_review_row
    sample_df = add_expert_bounding_boxes(load_or_create_pair_review_sample(section))
    model_sample_df = sample_df[sample_df["model_short"].eq(model)].sort_values("sample_index")
    if not 0 <= index < len(model_sample_df):
        raise IndexError(f"index must be between 0 and {len(model_sample_df) - 1}")

    row = model_sample_df.iloc[index]
    current_review_row = (section, row)

    print(f"{index + 1}/{len(model_sample_df)} {section} ({row['relation']}) - {model}")
    print("Pair IDs:", row.get("expert_comment_id"), "|", row["llm_comment_id"])
    print("Task:", row["task"])
    # The unsupported review is deliberately blind to the expert comment.
    if section != "unsupported":
        print("Expert comment:", row["expert_observed_issue"])
    print("LLM comment:", row["llm_observed_issue"])

    expert_bbox = row.get("expert_bounding_box")
    has_bbox = not pd.isna(expert_bbox)
    show_screen(
        screen_id=row["screen_id"],
        bboxes=[expert_bbox] if has_bbox else None,
        labels=["Expert comment"] if has_bbox else None,
        colors=["tab:red"],
        width=250,
    )


def save_current_pair_review(label, note=""):
    """Upsert one verdict for the pair most recently shown by show_comment_for_review."""
    section, row = current_review_row
    record = pd.DataFrame([{
        "section": section,
        "model_short": row["model_short"],
        "sample_index": int(row["sample_index"]),
        "screen_task_id": row["screen_task_id"],
        "expert_comment_id": row.get("expert_comment_id"),
        "llm_comment_id": row["llm_comment_id"],
        "relation": row["relation"],
        "label": label,
        "note": note,
    }])

    key_cols = ["section", "model_short", "sample_index"]
    if PAIR_REVIEW_LABELS_PATH.exists():
        saved_df = pd.read_csv(PAIR_REVIEW_LABELS_PATH, dtype={"section": str})
        missing_key_cols = [col for col in key_cols if col not in saved_df.columns]
        if missing_key_cols:
            raise ValueError(f"Existing review file is missing columns: {missing_key_cols}")
        saved_df["section"] = saved_df["section"].map(SECTION_ALIASES).fillna(saved_df["section"])
        record = pd.concat([saved_df, record], ignore_index=True)

    record = record.drop_duplicates(key_cols, keep="last").sort_values(key_cols).reset_index(drop=True)
    record.to_csv(PAIR_REVIEW_LABELS_PATH, index=False)

    done = int(record["section"].eq(section).sum())
    print(f"Saved {done}/{PAIR_REVIEW_TARGET} {section} reviews to {PAIR_REVIEW_LABELS_PATH}")


def review_progress():
    if not PAIR_REVIEW_LABELS_PATH.exists():
        return pd.DataFrame(columns=["section", "model_short", "reviewed", "remaining"])
    saved_df = pd.read_csv(PAIR_REVIEW_LABELS_PATH, dtype={"section": str})
    saved_df["section"] = saved_df["section"].map(SECTION_ALIASES).fillna(saved_df["section"])
    progress_df = (
        saved_df.groupby(["section", "model_short"]).size().rename("reviewed")
        .reindex(
            pd.MultiIndex.from_product([list(REVIEW_SECTIONS), MODEL_ORDER], names=["section", "model_short"]),
            fill_value=0,
        )
        .reset_index()
    )
    progress_df["remaining"] = REVIEW_SAMPLE_SIZE - progress_df["reviewed"]
    return progress_df


display(review_progress())

### 7.2 Matched Comment Analysis

Review pairs the judge labelled `equivalent`. The question is whether the two comments really
describe the same issue, or whether the label only reflects surface-level semantic similarity.

In [ ]:
show_comment_for_review("matched", "claude4_7", 0)

In [ ]:
matched_score = 2  # 0 = different issues, 1 = partial, 2 = genuinely equivalent
matched_note = ""
save_current_pair_review(matched_score, matched_note)

### 7.3 Unmatched Expert Comment Analysis

Examine expert comments whose best relation is only `no_match` / `no_relation` or `related_but_different`, including the expert comments that were not matched by all three models.

In [ ]:
UNMATCHED_EXPERT_RELATIONS = ["related_but_different", "no_match"]


def build_unmatched_expert_comments(relations=UNMATCHED_EXPERT_RELATIONS):
    model_count = len(MODEL_ORDER)
    unmatched_by_model_df = expert_coverage_df[expert_coverage_df["best_relation"].isin(relations)]

    counts_df = (
        unmatched_by_model_df.groupby([*MODEL_GROUP_COLS, "best_relation"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=relations, fill_value=0)
    )
    counts_df["not_matched"] = counts_df.sum(axis=1)
    counts_df["expert_comments"] = expert_coverage_df.groupby(MODEL_GROUP_COLS, observed=True).size()
    counts_df["not_matched_pct"] = (counts_df["not_matched"] / counts_df["expert_comments"] * 100).round(1)
    counts_df = (
        counts_df.reset_index()
        .sort_values("model_short", key=lambda s: s.map(MODEL_RANK))
        .reset_index(drop=True)
    )

    expert_cols = [
        "screen_task_id", "expert_comment_id", "screen_id", "app_category",
        "task", "expert_observed_issue", "expert_bounding_box",
    ]
    all_models_df = (
        unmatched_by_model_df.groupby(["screen_task_id", "expert_comment_id"], as_index=False)
        .agg(models_not_matched=("model_short", "nunique"))
        .query("models_not_matched == @model_count")
        .merge(
            metric_pairs_df[expert_cols].drop_duplicates(["screen_task_id", "expert_comment_id"]),
            on=["screen_task_id", "expert_comment_id"],
            how="left",
        )
        .sort_values(["app_category", "screen_task_id", "expert_comment_id"])
        .reset_index(drop=True)
    )

    return counts_df, all_models_df


def display_unmatched_expert_comment_review(comments_df, limit=None):
    rows = comments_df.head(limit) if limit else comments_df
    for i, row in enumerate(rows.itertuples(index=False), start=1):
        print("=" * 100)
        print(f"{i}. {row.app_category} | screen_id={row.screen_id} | screen_task_id={row.screen_task_id} | expert_comment_id={row.expert_comment_id}")
        print(f"Task: {row.task}")
        print(f"Expert comment: {row.expert_observed_issue}")

        for model in MODEL_ORDER:
            model_df = metric_pairs_df[
                metric_pairs_df["model_short"].eq(model)
                & metric_pairs_df["screen_task_id"].eq(row.screen_task_id)
                & metric_pairs_df["expert_comment_id"].eq(row.expert_comment_id)
            ]
            model_df = model_df.assign(relation_rank=model_df["relation"].map(RELATION_RANK)).sort_values(
                ["relation_rank", "confidence"], ascending=[True, False]
            )

            print(f"\n{MODEL_DISPLAY[model]}")
            for j, llm_row in enumerate(model_df.drop_duplicates("llm_comment_id").itertuples(index=False), start=1):
                print(f"  {j}. [{llm_row.relation}, confidence={llm_row.confidence:.2f}] {llm_row.llm_observed_issue}")
        print()


In [ ]:
unmatched_expert_counts_df, unmatched_by_all_models_df = build_unmatched_expert_comments()

print("Expert comments not matched per model")
display_df(unmatched_expert_counts_df)

print(f"Expert comments not matched by all three models: {len(unmatched_by_all_models_df):,}")
display_df(unmatched_by_all_models_df[[
    "screen_id", "screen_task_id", "app_category", "expert_comment_id", "expert_observed_issue"
]])


In [ ]:
sample_index = 0
sample_n = 30
sample_seed = 42

unmatched_review_sample_df = unmatched_by_all_models_df.sample(
    n=min(sample_n, len(unmatched_by_all_models_df)),
    random_state=sample_seed,
).reset_index(drop=True)

unmatched_expert_sample_path = QUALITATIVE_DIR / "unmatched_expert_review_sample.csv"
unmatched_review_sample_df.to_csv(unmatched_expert_sample_path, index=False)

print(f"Saved sampled expert IDs to {unmatched_expert_sample_path}")

if not 0 <= sample_index < len(unmatched_review_sample_df):
    raise IndexError(f"sample_index must be between 0 and {len(unmatched_review_sample_df) - 1}")

print(f"Showing unmatched expert comment {sample_index + 1}/{len(unmatched_review_sample_df)}")
display_unmatched_expert_comment_review(unmatched_review_sample_df.iloc[[sample_index]])
show_screen(
    screen_id=unmatched_review_sample_df.iloc[sample_index]["screen_id"],
    bboxes=unmatched_review_sample_df.iloc[sample_index]["expert_bounding_box"],
    labels=["Expert target"],
)


#### 7.3.1 Shared Expert-Comment Omissions

We qualitatively inspected a reproducible random sample of 30 unique expert comments that were not covered by any of the three models. Each comment was assessed against the complete interface screenshot, with the expert-provided bounding box treated as the target region rather than as a crop that excluded surrounding interaction context. This sample therefore characterizes shared blind spots across models; it is not an estimate of model-specific failure rates.

The review showed that an unmatched expert comment does not necessarily constitute a clear model failure. Twelve of the 30 comments described a specific, visually localized concern supported by the target region, including insufficient edge margins, small or tightly spaced controls, weak typographic hierarchy, low-contrast text, and ambiguous icons. These cases provide the strongest evidence of shared omissions. Nine comments were dependent on interaction context or design conventions that could not be resolved from a static screenshot alone. This group included claims about missing Back or Cancel controls, the need to scroll, and insufficient information about an application's purpose. Five comments were broad or subjective (for example, that colours were unappealing, the layout was basic, or the composition lacked balance), making exact semantic recovery unlikely even when a model identified more concrete problems in the same region. The remaining four appeared inconsistent with the visible evidence: two reported spelling or icon-detail defects not present in the target region, one claimed that an icon-rich screen lacked icons, and one characterized a simple heart symbol as overly detailed.

The model outputs also revealed several near misses hidden by the categorical coverage score. For example, models identified weak contrast and grid imbalance where the expert used the broader description of unappealing colours and a basic layout; they identified weak hierarchy in the same navigation drawer where the expert compared heading and menu typography; and they criticized the date picker's low contrast and unclear operation while omitting the expert's specific request for a Cancel action. Conversely, some highly localized concerns, such as edge margins, control size and spacing, and small low-contrast text, were not recovered despite being visually verifiable. Thus, disagreement arose from at least three sources: genuine omission of an expert-identified issue, different but defensible prioritization of another problem in the same target area, and noise or underspecification in the expert reference itself.

These findings qualify the interpretation of expert-comment recall. Treating every unmatched expert statement as a false negative would overstate model unreliability because the expert annotations are not uniformly specific, observable, or correct. At the same time, the repeated omission of concrete accessibility and interaction details shows that fluent critique generation does not guarantee comprehensive expert alignment. Reliability should therefore be reported along two axes: coverage of valid expert concerns and groundedness of the models' additional critiques. A manual audit should code reference validity, visual observability, semantic near misses, and model-specific omissions before unmatched comments are used as evidence of failure.

For the main qualitative analysis, the sampling unit should be the normalized usability issue rather than the raw comment string. Near-duplicate expert comments across screens should be clustered, with one randomly selected instance per cluster, to prevent frequent templates such as missing Back navigation from dominating a small sample. Cluster prevalence should nevertheless be retained and reported separately, or used as a sampling weight, because deduplication changes the estimand from the distribution of comments to the diversity of issue types. The 30-comment intersection sample is appropriate for analyzing shared blind spots. A separate model-stratified audit (for example, 30 unmatched comments per model, deduplicated within issue cluster and annotated for whether the other models covered the issue) is required for comparative claims about which model is more reliable.

### 7.4 Unsupported Comment Analysis
- determine whether unsupported LLM comments are legitimate new observations
- identify hallucinations
- distinguish vague-but-valid comments from unsupported claims
- check whether expert annotations missed something

In [ ]:
show_comment_for_review("unsupported", "gemini3_1pro", 29)

In [ ]:
hallucination_label = 0  # 1 = hallucination, 0 = not hallucination
hallucination_note = ""
save_current_pair_review(hallucination_label, hallucination_note)

### 7.5 Contradiction Analysis
Focus on:
- true contradiction
- LLM wrong
- expert wrong or incomplete
- both valid under different interpretations
- ambiguity, missing context, or relation extraction issue

In [ ]:
show_comment_for_review("contradiction", "gemini3_1pro", 17)

In [ ]:
contradiction_score = 0  # 0 = no, 1 = partial, 2 = yes
contradiction_comment = "Ambiguity in human annotation"
save_current_pair_review(contradiction_score, contradiction_comment)